In [ ]:
%pip install groq python-dotenv numpy tqdm datasets math-verify

   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 4.4 MB/s  0:00:00
   ---------------------------------------- 0.0/780.4 kB ? eta -:--:--
   ---------------------------------------- 780.4/780.4 kB 5.8 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   --------------------- ------------------ 2.1/4.0 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------  3.9/4.0 MB 10.6 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 8.4 MB/s  0:00:00

   ----------------------------------------  0/11 [antlr4-python3-runtime]
   ----------------------------------------  0/11 [antlr4-python3-runtime]
   ----------------------------------------  0/11 [antlr4-python3-runtime]
   ----------------------------------------  0/11 [antlr4-python3-runtime]
   --- ------------------------------------  1/11 [xxhash]
  Attempting uninstall: dill
   --- ------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-lsp-server 1.13.1 requires jedi<0.20.0,>=0.17.2, but you have jedi 0.20.0 which is incompatible.
spyder 6.1.0 requires jedi<0.20.0,>=0.17.2, but you have jedi 0.20.0 which is incompatible.
streamlit 1.51.0 requires packaging<26,>=20, but you have packaging 26.2 which is incompatible.


In [5]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset, concatenate_datasets

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any, Optional

load_dotenv()
random.seed(0)

if not os.getenv("GROQ_API_KEY"):
    raise ValueError(
        "GROQ_API_KEY를 찾을 수 없습니다. "
        ".env 파일이 prompting.ipynb와 같은 폴더에 있는지 확인하세요."
    )

client = Groq()

MODEL = "llama-3.1-8b-instant"

#### MATH 데이터셋 불러오기

- 평가: `HuggingFaceH4/MATH-500`
- few-shot 예시: `HuggingFaceH4/MATH`의 과목별 train split


In [6]:
# MATH 데이터 난이도 및 과목 필터링
TARGET_LEVELS = [1, 2, 3]

TARGET_SUBJECTS = [
    "Algebra",
    "Intermediate Algebra",
    "Number Theory",
    "Counting & Probability",
]

# 평가용 MATH-500
math_dataset = load_dataset("HuggingFaceH4/MATH-500")
math_test_raw = math_dataset["test"]

# few-shot 예시용 MATH train
TRAIN_CONFIGS = [
    "algebra",
    "intermediate_algebra",
    "number_theory",
    "counting_and_probability",
]

train_parts = []

for config in TRAIN_CONFIGS:
    ds = load_dataset(
        "HuggingFaceH4/MATH",
        config,
        split="train"
    )
    train_parts.append(ds)

math_train_raw = concatenate_datasets(train_parts)

print(sorted(set(math_train_raw["type"])))
print("raw train size:", len(math_train_raw))


['Algebra', 'Counting & Probability', 'Intermediate Algebra', 'Number Theory']
raw train size: 4679


In [7]:
## 데이터셋 전처리
def extract_last_boxed(text: str) -> Optional[str]:
    """문자열에서 마지막 \\boxed{...}의 내용을 추출합니다."""
    if not text:
        return None

    starts = [m.start() for m in re.finditer(r"\\boxed\s*\{", text)]
    if not starts:
        return None

    start = starts[-1]
    open_brace = text.find("{", start)
    depth = 0

    for idx in range(open_brace, len(text)):
        if text[idx] == "{":
            depth += 1
        elif text[idx] == "}":
            depth -= 1
            if depth == 0:
                return text[open_brace + 1:idx].strip()

    return None


def parse_level(level_value) -> Optional[int]:
    match = re.search(r"\d+", str(level_value))
    return int(match.group()) if match else None


def prepare_math_train_row(row):
    return {
        "question": row["problem"],
        "answer": extract_last_boxed(row["solution"]),
        "rationale": row["solution"],
        "subject": row["type"],
        "level_num": parse_level(row["level"]),
    }


math_train = math_train_raw.map(prepare_math_train_row)

math_train = math_train.filter(
    lambda row: (
        row["level_num"] in TARGET_LEVELS
        and row["subject"] in TARGET_SUBJECTS
        and row["answer"] is not None
    )
)

math_test = math_test_raw.filter(
    lambda row: (
        parse_level(row["level"]) in TARGET_LEVELS
        and row["subject"] in TARGET_SUBJECTS
    )
)

print("math_train size:", len(math_train))
print("math_test size:", len(math_test))
print("train levels:", sorted(set(math_train["level_num"])))
print("test levels:", sorted(set(parse_level(x) for x in math_test["level"])))


math_train size: 2132
math_test size: 146
train levels: [1, 2, 3]
test levels: [1, 2, 3]


In [37]:
import time
import re


def generate_response_using_Llama(
        prompt: str,
        model: str = MODEL,
        system_message: str = (
            "You are a careful mathematical problem solver."
        ),
        temperature: float = 0.0,
        max_completion_tokens: int = 550,
        max_retries: int = 10
    ):
    """
    Groq API를 호출한다.

    출력 토큰 수를 제한하고, 429 오류에는 대기 후 재시도한다.
    """

    for attempt in range(max_retries):
        try:
            chat_completion = client.chat.completions.create(
                messages=[
                    {
                        "role": "system",
                        "content": system_message
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                model=model,
                temperature=temperature,
                max_completion_tokens=max_completion_tokens,
                seed=2026,
                stream=False
            )

            return chat_completion.choices[0].message.content

        except Exception as e:
            error_message = str(e)

            is_rate_limit = (
                "429" in error_message
                or "rate limit" in error_message.lower()
            )

            if not is_rate_limit:
                print(f"API call error: {error_message}")
                return None

            if attempt == max_retries - 1:
                print(
                    "API 사용량 제한으로 모든 재시도가 실패했습니다."
                )
                return None

            retry_match = re.search(
                r"try again in\s+([\d.]+)s",
                error_message,
                flags=re.IGNORECASE
            )

            if retry_match:
                wait_seconds = float(retry_match.group(1)) + 1
            else:
                wait_seconds = min(
                    60,
                    5 * (attempt + 1)
                )

            print(
                f"429 사용량 제한 발생: "
                f"{wait_seconds:.1f}초 후 재시도합니다. "
                f"({attempt + 1}/{max_retries})"
            )

            time.sleep(wait_seconds)

    return None

#### 응답 잘 나오는지 확인하기

In [38]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello world. I'm ready to tackle any mathematical problem you'd like to throw at me. What's on your mind?


#### MATH 데이터셋 확인하기

In [10]:
print("[Question]")
print(math_test[0]["problem"])
print("=" * 100)
print("[Answer]")
print(math_test[0]["answer"])
print("=" * 100)
print("[Solution]")
print(math_test[0]["solution"])


[Question]
If $f(x) = \frac{3x-2}{x-2}$, what is the value of $f(-2) +f(-1)+f(0)$? Express your answer as a common fraction.
[Answer]
\frac{14}{3}
[Solution]
$f(-2)+f(-1)+f(0)=\frac{3(-2)-2}{-2-2}+\frac{3(-1)-2}{-1-2}+\frac{3(0)-2}{0-2}=\frac{-8}{-4}+\frac{-5}{-3}+\frac{-2}{-2}=2+\frac{5}{3}+1=\boxed{\frac{14}{3}}$


#### Utils 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [11]:
def extract_final_answer(response: str):
    """응답에서 마지막 \\boxed{...} 또는 Answer: 뒤의 답을 추출합니다."""
    if response is None:
        return None

    boxed_answer = extract_last_boxed(response)
    if boxed_answer is not None:
        return boxed_answer

    matches = re.findall(
        r"(?:Final Answer|Answer)\s*:\s*(.+)",
        response,
        re.IGNORECASE
    )
    if matches:
        return matches[-1].strip().strip("$")

    return None


def normalize_math_text(text: Any) -> str:
    text = str(text).strip().strip("$")
    text = text.replace(r"\displaystyle", "")
    text = text.replace(r"\dfrac", r"\frac")
    text = text.replace(r"\tfrac", r"\frac")
    text = text.replace(r"\,", "")
    text = text.replace(" ", "")
    return text.rstrip(".")


try:
    from math_verify import parse, verify
    MATH_VERIFY_AVAILABLE = True
except Exception:
    MATH_VERIFY_AVAILABLE = False


def answers_equivalent(
    correct_answer: str,
    predicted_answer: Optional[str]
) -> bool:
    if predicted_answer is None:
        return False

    if MATH_VERIFY_AVAILABLE:
        try:
            correct_parsed = parse(f"${correct_answer}$")
            predicted_parsed = parse(f"${predicted_answer}$")

            if verify(correct_parsed, predicted_parsed):
                return True
        except Exception:
            pass

    return (
        normalize_math_text(correct_answer)
        == normalize_math_text(predicted_answer)
    )


print("math-verify available:", MATH_VERIFY_AVAILABLE)


math-verify available: True


In [12]:
### 수정해도 됩니다!
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = MODEL,
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["problem"]
        correct_answer = str(dataset[i]["answer"]).strip()

        final_prompt = prompt.replace("{question}", question)

        response = generate_response_using_Llama(
            prompt=final_prompt,
            model=model
        )

        predicted_answer = (
            extract_final_answer(response)
            if response else None
        )
        is_correct = answers_equivalent(
            correct_answer,
            predicted_answer
        )

        if VERBOSE:
            print("=" * 50)
            print(response)
            print(f"Correct Answer: {correct_answer}")
            print(f"Predicted Answer: {predicted_answer}")
            print(f"Correct: {is_correct}")
            print("=" * 50)

        if is_correct:
            correct += 1

        total += 1

        results.append({
            "question": question,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct,
            "subject": dataset[i]["subject"],
            "level": parse_level(dataset[i]["level"]),
            "response": response,
        })

        if total % 5 == 0:
            current_accuracy = correct / total
            print(f"Progress: [{total}/{min(num_samples, len(dataset))}]")
            print(f"Current Acc.: [{current_accuracy:.2%}]")

    accuracy = correct / total if total > 0 else 0.0
    return results, accuracy


In [13]:
def save_final_result(
    results: List[Dict[str, Any]],
    accuracy: float,
    filename: str
) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += "[Details]\n"

    for idx, result in enumerate(results):
        result_str += f"Question {idx + 1}: {result['question']}\n"
        result_str += f"Subject: {result['subject']}\n"
        result_str += f"Level: {result['level']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)


#### 1. Direct Prompting with few-shot examples

In [26]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = math_train

    sampled_indices = random.sample(
        range(len(train_dataset)),
        num_examples
    )

    prompt = (
        "Instruction:\n"
        "Solve the following mathematical question and generate ONLY the final answer "
        "after the tag 'Answer:' without any rationale. "
        "Use valid mathematical notation.\n"
    )

    for idx, i in enumerate(sampled_indices):
        cur_question = train_dataset[i]["question"]
        cur_answer = train_dataset[i]["answer"]

        prompt += f"\n[Example {idx + 1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer: {cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt


In [15]:
### 어떤 방식으로 저장되는지 확인해보세요!

PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=math_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")
print(f"Direct 3-shot demo accuracy: {accuracy:.2%}")


 50%|█████     | 5/10 [00:02<00:02,  2.36it/s]

Progress: [5/10]
Current Acc.: [80.00%]


100%|██████████| 10/10 [00:04<00:00,  2.15it/s]

Progress: [10/10]
Current Acc.: [70.00%]
Direct 3-shot demo accuracy: 70.00%


In [27]:
# 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!
direct_accuracies = {}

for shot in [0, 3, 5]:
    print("=" * 80)
    print(f"Direct Prompting: {shot}-shot 시작")
    print("=" * 80)

    prompt = construct_direct_prompt(shot)

    results, accuracy = run_benchmark_test(
        dataset=math_test,
        prompt=prompt,
        model=MODEL,
        num_samples=50,
        VERBOSE=False
    )

    filename = f"direct_prompting_{shot}.txt"

    save_final_result(
        results=results,
        accuracy=accuracy,
        filename=filename
    )

    direct_accuracies[shot] = accuracy

    print(f"Direct {shot}-shot accuracy: {accuracy:.2%}")
    print(f"저장 완료: {filename}")

Direct Prompting: 0-shot 시작


  0%|          | 0/50 [00:00<?, ?it/s]

 10%|█         | 5/50 [00:02<00:17,  2.57it/s]

Progress: [5/50]
Current Acc.: [40.00%]


 20%|██        | 10/50 [00:04<00:22,  1.80it/s]

...

Progress: [40/50]
Current Acc.: [50.00%]


 90%|█████████ | 45/50 [06:07<00:26,  5.32s/it]

Progress: [45/50]
Current Acc.: [44.44%]


100%|██████████| 50/50 [06:55<00:00,  8.31s/it]

Progress: [50/50]
Current Acc.: [44.00%]
Direct 5-shot accuracy: 44.00%
저장 완료: direct_prompting_5.txt


#### 2. Chain-of-Thought Prompting with few-shot examples

```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 됩니다.

In [28]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = math_train

    sampled_indices = random.sample(
        range(len(train_dataset)),
        num_examples
    )

    prompt = (
        "Instruction:\n"
        "Solve the following mathematical problem step by step. "
        "Explain the reasoning clearly and logically. "
        "After completing the reasoning, write the final answer exactly once "
        "inside \\boxed{...}. "
        "Do not put intermediate results inside \\boxed{...}.\n"
    )

    for idx, i in enumerate(sampled_indices):
        cur_question = train_dataset[i]["question"]
        cur_rationale = train_dataset[i]["rationale"]

        prompt += f"\n[Example {idx + 1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Step-by-step solution:\n{cur_rationale}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt


In [29]:
# 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!
cot_accuracies = {}

for shot in [0, 3, 5]:
    print("=" * 80)
    print(f"CoT Prompting: {shot}-shot 시작")
    print("=" * 80)

    prompt = construct_CoT_prompt(shot)

    results, accuracy = run_benchmark_test(
        dataset=math_test,
        prompt=prompt,
        model=MODEL,
        num_samples=50,
        VERBOSE=False
    )

    filename = f"CoT_prompting_{shot}.txt"

    save_final_result(
        results=results,
        accuracy=accuracy,
        filename=filename
    )

    cot_accuracies[shot] = accuracy

    print(f"CoT {shot}-shot accuracy: {accuracy:.2%}")
    print(f"저장 완료: {filename}")

CoT Prompting: 0-shot 시작


 10%|█         | 5/50 [00:04<00:39,  1.14it/s]

Progress: [5/50]
Current Acc.: [60.00%]


 20%|██        | 10/50 [00:21<02:27,  3.68s/it]

Progress: [10/50]
Current Acc.: [50.00%]


...

429 사용량 제한 발생: 10.0초 후 재시도합니다. (1/10)
429 사용량 제한 발생: 20.0초 후 재시도합니다. (2/10)
429 사용량 제한 발생: 30.0초 후 재시도합니다. (3/10)
429 사용량 제한 발생: 40.0초 후 재시도합니다. (4/10)
429 사용량 제한 발생: 50.0초 후 재시도합니다. (5/10)


 98%|█████████▊| 49/50 [1:22:39<03:22, 202.27s/it]

429 사용량 제한 발생: 10.0초 후 재시도합니다. (1/10)
429 사용량 제한 발생: 20.0초 후 재시도합니다. (2/10)
429 사용량 제한 발생: 30.0초 후 재시도합니다. (3/10)
429 사용량 제한 발생: 40.0초 후 재시도합니다. (4/10)
429 사용량 제한 발생: 50.0초 후 재시도합니다. (5/10)
429 사용량 제한 발생: 60.0초 후 재시도합니다. (6/10)


100%|██████████| 50/50 [1:26:12<00:00, 103.45s/it]

Progress: [50/50]
Current Acc.: [66.00%]
CoT 5-shot accuracy: 66.00%
저장 완료: CoT_prompting_5.txt


#### 3. Construct your prompt + few shot examples
목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올리기!
- 세션때 배운 내용을 활용하거나 본인만의 풀이 과정을 만드는 등 자유롭게 진행해주시면 됩니다.
- 정답률은 Direct Prompting, CoT Prompting을 한 결과보다 높으면 됩니다. (0-shot, 3-shot, 5-shot 각각에서 모두 Direct Prompting과 CoT Prompting보다 높은 정답률을 달성하지 않더라도 감안하여 채점하겠습니다. 종합적으로 비교했을 때 본인이 설계한 프롬프트가 전반적으로 더 높은 성능을 보이는지를 기준으로 보겠습니다.)

In [35]:
def select_my_examples(
        train_dataset,
        num_examples: int
    ):
    """
    train 데이터에서 과목별로 다양하고,
    지나치게 길지 않은 풀이 예시를 선택한다.

    test 데이터는 사용하지 않는다.
    """

    if num_examples == 0:
        return []

    subject_order = [
        "Algebra",
        "Intermediate Algebra",
        "Number Theory",
        "Counting & Probability",
        "Algebra",
    ]

    selected_indices = []
    used_indices = set()

    for subject in subject_order:
        candidates = []

        for i, row in enumerate(train_dataset):
            if i in used_indices:
                continue

            if row["subject"] != subject:
                continue

            rationale = str(row["rationale"])
            answer = str(row["answer"])

            # 지나치게 길거나 너무 짧은 풀이 제외
            if not 150 <= len(rationale) <= 900:
                continue

            # 지나치게 긴 정답 제외
            if len(answer) > 50:
                continue

            candidates.append(i)

        if candidates:
            # 항상 동일한 예시가 선택되도록 고정
            selected_index = candidates[
                len(selected_indices) % len(candidates)
            ]

            selected_indices.append(selected_index)
            used_indices.add(selected_index)

        if len(selected_indices) == num_examples:
            break

    if len(selected_indices) < num_examples:
        raise ValueError(
            "조건에 맞는 train 예시가 부족합니다."
        )

    return selected_indices

In [39]:
def construct_my_prompt(
        num_examples: int = 3
    ) -> str:
    """
    One-Pass Plan-Solve-Verify Prompting
    한 번의 API 응답 안에서 계획, 풀이, 검산을 모두 수행한다.
    """

    train_dataset = math_train

    sampled_indices = select_my_examples(
        train_dataset=train_dataset,
        num_examples=num_examples
    )

    prompt = (
        "You are an accurate mathematics competition solver.\n\n"

        "Solve each problem using this internal process:\n"
        "1. Identify exactly what is requested and all constraints.\n"
        "2. Choose the simplest reliable method.\n"
        "3. Calculate with exact values.\n"
        "4. Independently check the result by substitution, reverse "
        "calculation, estimation, or another method.\n"
        "5. Correct any arithmetic, sign, domain, root, counting, "
        "base-conversion, or formatting error before answering.\n\n"

        "Keep the reasoning concise: no more than 250 words.\n"
        "End with exactly one final answer inside \\boxed{...}.\n"
        "Put only the requested answer inside the box.\n"
        "Use \\text{...} for text answers.\n"
        "Put all required solutions in one box.\n"
        "Do not write anything after the box.\n"
    )

    for idx, i in enumerate(sampled_indices):
        question = str(
            train_dataset[i]["question"]
        )
        rationale = str(
            train_dataset[i]["rationale"]
        )
        answer = str(
            train_dataset[i]["answer"]
        )

        concise_rationale = rationale[:350]

        prompt += f"\n[Example {idx + 1}]\n"
        prompt += f"Problem:\n{question}\n"
        prompt += f"Concise reasoning:\n{concise_rationale}\n"
        prompt += f"Final answer: \\boxed{{{answer}}}\n"

    prompt += (
        "\n[New problem]\n"
        "Problem:\n{question}\n\n"
        "Give a concise solution, verify it, and end with the boxed answer:\n"
    )

    return prompt

In [33]:
def run_my_refinement_benchmark(
        dataset,
        prompt: str,
        model: str = MODEL,
        num_samples: int = 50,
        VERBOSE: bool = False,
        delay_seconds: float = 4.0
    ):
    """
    1차 풀이와 독립적인 2차 풀이를 생성한 뒤,
    2차 풀이를 최종 답으로 사용한다.
    """

    correct = 0
    total = 0
    results = []

    sample_count = min(
        num_samples,
        len(dataset)
    )

    for i in tqdm(range(sample_count)):
        question = dataset[i]["problem"]
        correct_answer = str(
            dataset[i]["answer"]
        ).strip()

        # 1차: few-shot 프롬프트를 이용한 풀이
        draft_prompt = prompt.replace(
            "{question}",
            question
        )

        draft_response = generate_response_using_Llama(
            prompt=draft_prompt,
            model=model,
            system_message=(
                "You are a rigorous mathematics competition solver. "
                "Use exact mathematics and verify your result."
            ),
            temperature=0.0
        )

        if draft_response is None:
            raise RuntimeError(
                f"{i + 1}번 문제의 첫 번째 API 호출이 실패했습니다."
            )

        time.sleep(delay_seconds)

        # 2차: 초안을 따라 하지 않고 문제를 독립적으로 다시 풀이
        verification_prompt = f"""
Solve the following mathematics problem independently from the beginning.

Do not trust or copy any previous proposed answer.
First derive your own answer using the most reliable method.

Mandatory checks:
1. Identify exactly what quantity the problem asks for.
2. Check arithmetic and algebra line by line.
3. Check domains, signs, constraints, and extraneous solutions.
4. If the problem asks for all roots, confirm that no root is missing.
5. If it is a counting problem, verify whether order and repetition matter.
6. If it uses another number base, convert and verify the final base notation.
7. If it is an optimization problem, check both feasibility and optimality.
8. Substitute or reverse-calculate the result whenever possible.

Problem:
{question}

After completing the independent solution, compare it with this first attempt:
{draft_response}

The first attempt may be wrong. Keep your independently derived answer unless
the comparison reveals a concrete error in your derivation.

Output requirements:
- End with exactly one final answer inside \\boxed{{...}}.
- Put only the requested answer inside the box.
- Use \\text{{...}} for text answers.
- Put every required solution inside one box.
- Do not write anything after the box.
"""

        final_response = generate_response_using_Llama(
            prompt=verification_prompt,
            model=model,
            system_message=(
                "You are an independent mathematical auditor. "
                "Recompute the problem rather than trusting the draft."
            ),
            temperature=0.0
        )

        if final_response is None:
            raise RuntimeError(
                f"{i + 1}번 문제의 검산 API 호출이 실패했습니다."
            )

        predicted_answer = extract_final_answer(
            final_response
        )

        is_correct = answers_equivalent(
            correct_answer,
            predicted_answer
        )

        if is_correct:
            correct += 1

        total += 1

        results.append({
            "question": question,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct,
            "subject": dataset[i]["subject"],
            "level": parse_level(
                dataset[i]["level"]
            ),
            "response": final_response,
            "draft_response": draft_response,
        })

        if VERBOSE:
            print("=" * 80)
            print("Question:")
            print(question)
            print("\nFirst attempt:")
            print(draft_response)
            print("\nIndependent verification:")
            print(final_response)
            print("\nCorrect answer:", correct_answer)
            print("Predicted answer:", predicted_answer)
            print("Correct:", is_correct)

        if total % 5 == 0:
            current_accuracy = correct / total

            print(f"Progress: [{total}/{sample_count}]")
            print(
                f"Current Acc.: "
                f"[{current_accuracy:.2%}]"
            )

        time.sleep(delay_seconds)

    accuracy = (
        correct / total
        if total > 0
        else 0.0
    )

    return results, accuracy

In [40]:
def run_my_fast_benchmark(
        dataset,
        prompt: str,
        model: str = MODEL,
        num_samples: int = 50,
        VERBOSE: bool = False,
        delay_seconds: float = 3.0
    ):
    """
    문제당 API를 한 번만 호출한다.
    모델이 한 응답 안에서 풀이와 검산을 모두 수행한다.
    """

    correct = 0
    total = 0
    results = []

    sample_count = min(
        num_samples,
        len(dataset)
    )

    for i in tqdm(range(sample_count)):
        question = dataset[i]["problem"]
        correct_answer = str(
            dataset[i]["answer"]
        ).strip()

        final_prompt = prompt.replace(
            "{question}",
            question
        )

        response = generate_response_using_Llama(
            prompt=final_prompt,
            model=model,
            system_message=(
                "You are a rigorous mathematics solver. "
                "Solve, verify, and return one boxed final answer."
            ),
            temperature=0.0,
            max_completion_tokens=550
        )

        if response is None:
            raise RuntimeError(
                f"{i + 1}번 문제의 API 호출이 실패했습니다. "
                "결과 파일을 저장하지 않습니다."
            )

        predicted_answer = extract_final_answer(
            response
        )

        is_correct = answers_equivalent(
            correct_answer,
            predicted_answer
        )

        if is_correct:
            correct += 1

        total += 1

        results.append({
            "question": question,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct,
            "subject": dataset[i]["subject"],
            "level": parse_level(
                dataset[i]["level"]
            ),
            "response": response,
        })

        if VERBOSE:
            print("=" * 80)
            print("Question:")
            print(question)
            print("\nResponse:")
            print(response)
            print("\nCorrect answer:", correct_answer)
            print("Predicted answer:", predicted_answer)
            print("Correct:", is_correct)

        if total % 5 == 0:
            current_accuracy = correct / total

            print(f"Progress: [{total}/{sample_count}]")
            print(
                f"Current Acc.: "
                f"[{current_accuracy:.2%}]"
            )

        time.sleep(delay_seconds)

    accuracy = (
        correct / total
        if total > 0
        else 0.0
    )

    return results, accuracy

In [41]:
my_accuracies = {}

for shot in [0, 3, 5]:
    print("=" * 80)
    print(f"My Prompting: {shot}-shot 시작")
    print("=" * 80)

    prompt = construct_my_prompt(shot)

    results, accuracy = run_my_fast_benchmark(
        dataset=math_test,
        prompt=prompt,
        model=MODEL,
        num_samples=50,
        VERBOSE=False,
        delay_seconds=3.0
    )

    filename = f"My_prompting_{shot}.txt"

    save_final_result(
        results=results,
        accuracy=accuracy,
        filename=filename
    )

    my_accuracies[shot] = accuracy

    print(
        f"My Prompting {shot}-shot accuracy: "
        f"{accuracy:.2%}"
    )
    print(f"저장 완료: {filename}")

My Prompting: 0-shot 시작


  2%|▏         | 1/50 [00:03<03:05,  3.79s/it]

429 사용량 제한 발생: 5.0초 후 재시도합니다. (1/10)
429 사용량 제한 발생: 10.0초 후 재시도합니다. (2/10)
429 사용량 제한 발생: 15.0초 후 재시도합니다. (3/10)
429 사용량 제한 발생: 20.0초 후 재시도합니다. (4/10)


  4%|▍         | 2/50 [02:10<1:01:00, 76.27s/it]

429 사용량 제한 발생: 5.0초 후 재시도합니다. (1/10)
429 사용량 제한 발생: 10.0초 후 재시도합니다. (2/10)
429 사용량 제한 발생: 15.0초 후 재시도합니다. (3/10)
429 사용량 제한 발생: 20.0초 후 재시도합니다. (4/10)
429 사용량 제한 발생: 25.0초 후 재시도합니다. (5/10)
429 사용량 제한 발생: 30.0초 후 재시도합니다. (6/10)
429 사용량 제한 발생: 35.0초 후 재시도합니다. (7/10)


...

  4%|▍         | 2/50 [09:32<3:48:35, 285.73s/it]

429 사용량 제한 발생: 5.0초 후 재시도합니다. (1/10)
429 사용량 제한 발생: 10.0초 후 재시도합니다. (2/10)
429 사용량 제한 발생: 15.0초 후 재시도합니다. (3/10)
429 사용량 제한 발생: 20.0초 후 재시도합니다. (4/10)
429 사용량 제한 발생: 25.0초 후 재시도합니다. (5/10)
429 사용량 제한 발생: 30.0초 후 재시도합니다. (6/10)
429 사용량 제한 발생: 35.0초 후 재시도합니다. (7/10)
429 사용량 제한 발생: 40.0초 후 재시도합니다. (8/10)
429 사용량 제한 발생: 45.0초 후 재시도합니다. (9/10)


  4%|▍         | 2/50 [13:19<5:19:45, 399.69s/it]

API 사용량 제한으로 모든 재시도가 실패했습니다.


RuntimeError: 3번 문제의 API 호출이 실패했습니다. 결과 파일을 저장하지 않습니다.

In [42]:
# My Prompting 5-shot만 다시 실행

shot = 5

print("=" * 80)
print("My Prompting: 5-shot 재실행")
print("=" * 80)

prompt = construct_my_prompt(shot)

my_5_results, my_5_accuracy = run_my_fast_benchmark(
    dataset=math_test,
    prompt=prompt,
    model=MODEL,
    num_samples=50,
    VERBOSE=False,
    delay_seconds=8.0
)

print(f"\nMy Prompting 5-shot accuracy: {my_5_accuracy:.2%}")

successful_count = sum(
    result["response"] is not None
    for result in my_5_results
)

print(f"정상 응답 수: {successful_count}/50")

if successful_count != 50:
    raise RuntimeError(
        "50문제 중 일부 API 응답이 실패했습니다. "
        "결과 파일을 저장하지 않습니다."
    )

save_final_result(
    results=my_5_results,
    accuracy=my_5_accuracy,
    filename="My_prompting_5.txt"
)

print("My_prompting_5.txt 저장 완료")

My Prompting: 5-shot 재실행


  8%|▊         | 4/50 [00:35<06:44,  8.79s/it]

Progress: [5/50]
Current Acc.: [60.00%]


 18%|█▊        | 9/50 [01:45<09:45, 14.27s/it]

Progress: [10/50]
Current Acc.: [70.00%]


...

Progress: [45/50]
Current Acc.: [60.00%]


 98%|█████████▊| 49/50 [10:58<00:15, 15.27s/it]

Progress: [50/50]
Current Acc.: [62.00%]


100%|██████████| 50/50 [11:11<00:00, 13.43s/it]


My Prompting 5-shot accuracy: 62.00%
정상 응답 수: 50/50
My_prompting_5.txt 저장 완료


In [46]:
my_accuracies[5] = my_5_accuracy

print(my_accuracies)

{0: 0.68, 3: 0.62, 5: 0.62}


In [47]:
import pandas as pd

accuracy_table = pd.DataFrame({
    "Direct Prompting": direct_accuracies,
    "CoT Prompting": cot_accuracies,
    "My Prompting": my_accuracies
}).T

accuracy_table.columns = [
    f"{shot}-shot"
    for shot in accuracy_table.columns
]

accuracy_table

,0-shot,3-shot,5-shot
Direct Prompting,0.26,0.38,0.44
CoT Prompting,0.60,0.62,0.66
My Prompting,0.68,0.62,0.62


In [48]:
direct_mean = sum(direct_accuracies.values()) / len(direct_accuracies)
cot_mean = sum(cot_accuracies.values()) / len(cot_accuracies)
my_mean = sum(my_accuracies.values()) / len(my_accuracies)

print(f"Direct 평균: {direct_mean:.2%}")
print(f"CoT 평균: {cot_mean:.2%}")
print(f"My 평균: {my_mean:.2%}")

Direct 평균: 36.00%
CoT 평균: 62.67%
My 평균: 64.00%


### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot 정답률을 표로 보여주세요.
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요.
3. 본인이 작성한 프롬프트 기법에 대해서 설명하고 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요.
4. 위 내용들을 `PROMPTING.md`에 보고서로 작성해주세요.
